### Разведка каталога, схем и физических размеров таблиц

Проверяем доступные схемы в `lakehouse` (`raw`, `stage`, `mart`), список таблиц, их структуру (`printSchema`), сэмплы и объём в Iceberg (`.files`).

**Выводы:**
* `mongo__tower_damage_events`: 49 файлов, 1 552,38 МБ, 241 654 288 строк.
* `mongo__building_events`: 18 файлов, 349,89 МБ, 49 839 696 строк.
* `pg__match_players`: 48 файлов, 161,60 МБ, 4 651 461 строка.
* `pg__matches`: 52 файла, 7,49 МБ, 315 079 строк.
* Поля `byability`, `byitem`, `fromnpc` в таблице фактов имеют тип `integer` и кодируют отсутствие значения нулём.
* `steam_account_id` в сэмпле 32-битный.

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext("Tables") as spark:

    # 1. Проверка доступных пространств имен (схем) в каталоге lakehouse
    print("=== NAMESPACES IN lakehouse ===")
    spark.sql("SHOW NAMESPACES IN lakehouse").show(truncate=False)
    
    print("=== TABLES IN lakehouse.raw ===")
    spark.sql("SHOW TABLES IN lakehouse.raw").show(100, truncate=False)
    
    # 2. Список целевых таблиц из плана
    target_tables = [
        "lakehouse.raw.mongo__tower_damage_events",
        "lakehouse.raw.mongo__building_events",
        "lakehouse.raw.pg__match_players",
        "lakehouse.raw.pg__matches",
        "lakehouse.raw.pg__heroes",
        "lakehouse.raw.pg__abilities",
        "lakehouse.raw.pg__hero_abilities",
        "lakehouse.raw.pg__npcs",
        "lakehouse.raw.pg__items",
        "lakehouse.raw.pg__game_modes",
        "lakehouse.raw.pg__lobby_types",
        "lakehouse.raw.mongo__tower_death_events"
    ]
    
    # 3. Инспекция схем, сэмплов и физических размеров (через Iceberg metadata table .files)
    for tbl in target_tables:
        print(f"\n{'='*60}")
        print(f"TABLE: {tbl}")
        try:
            df = spark.table(tbl)
            df.printSchema()
            df.show(5, truncate=False)
            
            # Проверка размера и количества файлов в Iceberg
            try:
                stats = spark.sql(f"""
                    SELECT 
                        COUNT(*) AS files_count,
                        ROUND(SUM(file_size_in_bytes) / 1024.0 / 1024.0, 2) AS size_mb,
                        SUM(record_count) AS total_records_meta
                    FROM {tbl}.files
                """)
                stats.show(truncate=False)
            except Exception as e_meta:
                print(f"[Meta .files unavailable]: {e_meta}")
                
        except Exception as e:
            print(f"[ERROR reading {tbl}]: {e}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 05:15:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


=== NAMESPACES IN lakehouse ===
+---------+
|namespace|
+---------+
|raw      |
|stage    |
|mart     |
+---------+

=== TABLES IN lakehouse.raw ===
+---------+-----------------------------------+-----------+
|namespace|tableName                          |isTemporary|
+---------+-----------------------------------+-----------+
|raw      |mongo__item_used_events            |false      |
|raw      |mongo__ability_learn_events        |false      |
|raw      |pg__lobby_types                    |false      |
|raw      |mongo__building_events             |false      |
|raw      |mongo__kill_events                 |false      |
|raw      |pg__neutral_items                  |false      |
|raw      |pg__npcs                           |false      |
|raw      |mongo__assist_events               |false      |
|raw      |mongo__player_update_level_events  |false      |
|raw      |pg__players                        |false      |
|raw      |mongo__purchase_events             |false      |
|raw      |

+--------+---------+------+------+-------+------------------------+----------+-----+----------------+----+------------------------------------+--------------------------+-----------------------+
|attacker|byability|byitem|damage|fromnpc|id                      |match_id  |npcid|steam_account_id|time|job_id                              |sys_insert_stamp          |src_modified_stamp     |
+--------+---------+------+------+-------+------------------------+----------+-----+----------------+----+------------------------------------+--------------------------+-----------------------+
|75      |0        |0     |28    |0      |6ad5517d5ce156150e08ef04|9044689256|29   |301865375       |1183|53e30e02-34a8-4899-a0c7-ab32942d6462|2026-10-19 00:18:44.795731|2026-10-18 23:08:45.209|
|75      |0        |0     |27    |0      |6ad5517d5ce156150e08ef05|9044689256|29   |301865375       |1184|53e30e02-34a8-4899-a0c7-ab32942d6462|2026-10-19 00:18:44.795731|2026-10-18 23:08:45.209|
|75      |0        |0    

+-----------+-------+------------------+
|files_count|size_mb|total_records_meta|
+-----------+-------+------------------+
|49         |1552.38|241654288         |
+-----------+-------+------------------+


TABLE: lakehouse.raw.mongo__building_events
root
 |-- id: string (nullable = false)
 |-- didshrineactivate: boolean (nullable = true)
 |-- hp: integer (nullable = true)
 |-- indexid: integer (nullable = true)
 |-- isradiant: boolean (nullable = true)
 |-- match_id: long (nullable = true)
 |-- maxhp: integer (nullable = true)
 |-- npcid: integer (nullable = true)
 |-- positionx: integer (nullable = true)
 |-- positiony: integer (nullable = true)
 |-- time: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- job_id: string (nullable = true)
 |-- sys_insert_stamp: timestamp (nullable = true)
 |-- src_modified_stamp: timestamp (nullable = true)

+------------------------+-----------------+----+-------+---------+----------+-----+-----+---------+---------+----+-----+-------

26/11/13 05:16:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+----------------+--------------+-------+----------+----------+----------+------+------+----+------------+----------+-----------+------------+-------+----------+------+------+------+------+------+------+------------+----+-----+---------+-----+----------+---------+-----------+----+------------+-----+----------+------------------------------------+--------------------------+--------------------------+
|account_id|aghanims_scepter|aghanims_shard|assists|backpack_0|backpack_1|backpack_2|deaths|denies|gold|gold_per_min|gold_spent|hero_damage|hero_healing|hero_id|is_radiant|item_0|item_1|item_2|item_3|item_4|item_5|item_neutral|kda |kills|last_hits|level|match_id  |net_worth|player_slot|rank|tower_damage|win  |xp_per_min|job_id                              |sys_insert_stamp          |src_modified_stamp        |
+----------+----------------+--------------+-------+----------+----------+----------+------+------+----+------------+----------+-----------+------------+-------+----------+

+---------------------+----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+-------+------------------+---------------------------------------------------------------------------------+------------------+---------+------------------------------------+--------------------------+------------------+
|components           |cost|description                                                                                                                                                                                                                                                                                                                                                        

+--------+--------+---------------------+------------------------------------+--------------------------+------------------+
|balanced|lobby_id|name                 |job_id                              |sys_insert_stamp          |src_modified_stamp|
+--------+--------+---------------------+------------------------------------+--------------------------+------------------+
|true    |0       |lobby_type_normal    |3d28790c-ae20-4a2f-925c-e88d000f296d|2026-10-18 16:28:08.506189|NULL              |
|true    |1       |lobby_type_practice  |3d28790c-ae20-4a2f-925c-e88d000f296d|2026-10-18 16:28:08.506189|NULL              |
|true    |2       |lobby_type_tournament|3d28790c-ae20-4a2f-925c-e88d000f296d|2026-10-18 16:28:08.506189|NULL              |
|false   |3       |lobby_type_tutorial  |3d28790c-ae20-4a2f-925c-e88d000f296d|2026-10-18 16:28:08.506189|NULL              |
|false   |4       |lobby_type_coop_bots |3d28790c-ae20-4a2f-925c-e88d000f296d|2026-10-18 16:28:08.506189|NULL              |


+----+------------------------+----------+-------+----+------------------------------------+--------------------------+-----------------------+
|dire|id                      |match_id  |radiant|time|job_id                              |sys_insert_stamp          |src_modified_stamp     |
+----+------------------------+----------+-------+----+------------------------------------+--------------------------+-----------------------+
|2   |6ad526fd614f616b5c154a5e|9044681832|0      |467 |8843dc22-7fc5-4ec9-87b0-6aa93341b8af|2026-10-18 21:11:46.599861|2026-10-18 20:07:24.937|
|3   |6ad526fd614f616b5c154a5f|9044681832|0      |620 |8843dc22-7fc5-4ec9-87b0-6aa93341b8af|2026-10-18 21:11:46.599861|2026-10-18 20:07:24.937|
|7   |6ad526fd614f616b5c154a60|9044681832|0      |753 |8843dc22-7fc5-4ec9-87b0-6aa93341b8af|2026-10-18 21:11:46.599861|2026-10-18 20:07:24.937|
|7   |6ad526fd614f616b5c154a61|9044681832|2      |758 |8843dc22-7fc5-4ec9-87b0-6aa93341b8af|2026-10-18 21:11:46.599861|2026-10-18 20:07:

### Аудит `pg__matches` и `pg__match_players`

Проверяем уникальность матчей, валидность длительности (`duration`), распределение режимов игры (`game_mode`) и причину кратности ~14,76 строк на матч в `pg__match_players`.

**Выводы:**
* В `pg__matches`: 315 079 уникальных `match_id`, дубликатов и записей с `duration <= 0` нет.
* Есть существенная неоднородность режимов: 101 497 матчей (32,2%) имеют `game_mode = 23` (Turbo), 201 235 матчей - `game_mode = 22` (All Draft).
* В `pg__match_players`: 4 651 461 строка и 465 155 уникальных `match_id`. Дубликатов по `(match_id, player_slot)` нет, но 150 444 матча отсутствуют в `pg__matches`.
* Зафиксированы аномалии в `pg__match_players`: 76 619 строк с `account_id <= 0` и 9 597 строк с `hero_id <= 0`.

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext("pg__match_players_audit") as spark:
    
    # Аудит pg__matches и строк в pg__match_players
    print(f"\n{'='*60}\nAUDIT: pg__matches & pg__match_players")
    matches_df = spark.table("lakehouse.raw.pg__matches").select(
        "match_id", "duration", "game_mode", "lobby_type", "human_players", "radiant_win"
    )
    mp_df = spark.table("lakehouse.raw.pg__match_players").select(
        "match_id", "account_id", "player_slot", "hero_id", "is_radiant", "win", "tower_damage"
    )
    
    matches_df.agg(
        F.count("*").alias("rows_cnt"),
        F.countDistinct("match_id").alias("distinct_match_id"),
        F.sum(F.when(F.col("duration").isNull() | (F.col("duration") <= 0), 1).otherwise(0)).alias("bad_duration")
    ).show(truncate=False)
    
    print("Top game_mode / human_players / lobby_type in pg__matches:")
    matches_df.groupBy("game_mode", "lobby_type", "human_players").count().orderBy(F.desc("count")).show(20, truncate=False)
    
    mp_df.agg(
        F.count("*").alias("mp_rows_cnt"),
        F.countDistinct("match_id").alias("mp_distinct_matches"),
        F.sum(F.when(F.col("account_id").isNull(), 1).otherwise(0)).alias("null_account_id"),
        F.sum(F.when(F.col("account_id") <= 0, 1).otherwise(0)).alias("nonpos_account_id"),
        F.sum(F.when(F.col("hero_id").isNull() | (F.col("hero_id") <= 0), 1).otherwise(0)).alias("bad_hero_id")
    ).show(truncate=False)
    
    print("Проверка дубликатов (match_id, player_slot) в pg__match_players:")
    mp_df.groupBy("match_id", "player_slot").count().filter(F.col("count") > 1).agg(
        F.count("*").alias("dup_match_slot_pairs")
    ).show()
    
    print("Сколько match_id из pg__match_players отсутствуют в pg__matches:")
    print(mp_df.select("match_id").distinct().join(
        F.broadcast(matches_df.select("match_id").distinct()), on="match_id", how="left_anti"
    ).count())


AUDIT: pg__matches & pg__match_players


+--------+-----------------+------------+
|rows_cnt|distinct_match_id|bad_duration|
+--------+-----------------+------------+
|315079  |315079           |0           |
+--------+-----------------+------------+

Top game_mode / human_players / lobby_type in pg__matches:


+---------+----------+-------------+------+
|game_mode|lobby_type|human_players|count |
+---------+----------+-------------+------+
|22       |7         |10           |173359|
|23       |0         |10           |101497|
|22       |0         |10           |27863 |
|4        |0         |10           |11599 |
|3        |0         |10           |506   |
|2        |1         |10           |195   |
|16       |1         |10           |40    |
|22       |1         |10           |13    |
|3        |1         |10           |4     |
|4        |1         |10           |2     |
|1        |1         |10           |1     |
+---------+----------+-------------+------+



+-----------+-------------------+---------------+-----------------+-----------+
|mp_rows_cnt|mp_distinct_matches|null_account_id|nonpos_account_id|bad_hero_id|
+-----------+-------------------+---------------+-----------------+-----------+
|4651461    |465155             |0              |76619            |9597       |
+-----------+-------------------+---------------+-----------------+-----------+

Проверка дубликатов (match_id, player_slot) в pg__match_players:


+--------------------+
|dup_match_slot_pairs|
+--------------------+
|                   0|
+--------------------+

Сколько match_id из pg__match_players отсутствуют в pg__matches:


150444


### Аудит качества `mongo__tower_damage_events`

Профилируем 241 654 288 событий за один проход: пропуски, граничные значения (`damage <= 0`, `time < 0`), разрядность аккаунтов, источники урона и домены `npcid`, `fromnpc`.

**Выводы:**
* Пропусков, неположительного урона, отрицательного времени, 64-битных `steam_account_id`, некорректных `attacker` и `npcid` не обнаружено.
* Общий урон: 10 060 397 786. Из них `byability > 0` — 35 603 110 событий (773 734 560 урона), `byitem > 0` — 1 514 926 событий (94 800 472 урона), одновременно оба поля не заполнены ни разу (0).
* Выявлено 44 609 533 события с `fromnpc > 0` (урон от не-геройских юнитов).
* Оценка `approx_count_distinct(id)` дала 225 552 730 против 241 654 288 строк (требуется точная проверка дубликатов по `id`).
* В топе `npcid` присутствуют ID 50, 51, 39, 45 и др. Необходимо проверить, какие из них являются башнями, а какие другими строениями.

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext("mongo__tower_damage_events_audit") as spark:

    # Однопроходный аудит mongo__tower_damage_events
    print(f"\n{'='*60}\nAUDIT: mongo__tower_damage_events (Single-Pass)")
    tde_df = spark.table("lakehouse.raw.mongo__tower_damage_events").select(
        "id", "match_id", "time", "damage", "attacker", "byability", "byitem", "fromnpc", "npcid", "steam_account_id"
    )
    
    tde_df.agg(
        F.count("*").alias("total_rows"),
        F.approx_count_distinct("id").alias("approx_distinct_id"),
        F.countDistinct("match_id").alias("distinct_matches"),
        
        # Пропуски и невалидные значения
        F.sum(F.when(F.col("match_id").isNull(), 1).otherwise(0)).alias("null_match_id"),
        F.sum(F.when(F.col("damage").isNull(), 1).otherwise(0)).alias("null_damage"),
        F.sum(F.when(F.col("damage") <= 0, 1).otherwise(0)).alias("nonpos_damage"),
        F.sum(F.when(F.col("time").isNull(), 1).otherwise(0)).alias("null_time"),
        F.sum(F.when(F.col("time") < 0, 1).otherwise(0)).alias("negative_time"),
        
        # Проверка идентификаторов и источников урона
        F.sum(F.when(F.col("steam_account_id").isNull(), 1).otherwise(0)).alias("null_steam_acc"),
        F.sum(F.when(F.col("steam_account_id") <= 0, 1).otherwise(0)).alias("nonpos_steam_acc"),
        F.sum(F.when(F.col("steam_account_id") > 4294967295, 1).otherwise(0)).alias("steam_acc_64bit"),
        
        F.sum(F.when(F.col("byability").isNull(), 1).otherwise(0)).alias("null_byability"),
        F.sum(F.when(F.col("byability") > 0, 1).otherwise(0)).alias("has_byability"),
        F.sum(F.when(F.col("byitem") > 0, 1).otherwise(0)).alias("has_byitem"),
        F.sum(F.when((F.col("byability") > 0) & (F.col("byitem") > 0), 1).otherwise(0)).alias("both_ability_and_item"),
        
        F.sum(F.when(F.col("fromnpc") > 0, 1).otherwise(0)).alias("has_fromnpc"),
        F.sum(F.when(F.col("attacker").isNull() | (F.col("attacker") <= 0), 1).otherwise(0)).alias("bad_attacker"),
        F.sum(F.when(F.col("npcid").isNull() | (F.col("npcid") <= 0), 1).otherwise(0)).alias("bad_npcid"),
        
        # Объем урона по источникам
        F.sum(F.col("damage").cast("long")).alias("total_damage_sum"),
        F.sum(F.when(F.col("byability") > 0, F.col("damage")).otherwise(0).cast("long")).alias("ability_damage_sum"),
        F.sum(F.when(F.col("byitem") > 0, F.col("damage")).otherwise(0).cast("long")).alias("item_damage_sum")
    ).show(truncate=False)
    
    # Топы значений для понимания доменов npcid, attacker, fromnpc
    print("Top 25 npcid in tower_damage_events:")
    tde_df.groupBy("npcid").agg(
        F.count("*").alias("events"),
        F.sum("damage").alias("dmg")
    ).orderBy(F.desc("events")).show(25, truncate=False)
    
    print("Top 25 attacker where steam_account_id is null or <= 0:")
    tde_df.filter(F.col("steam_account_id").isNull() | (F.col("steam_account_id") <= 0)) \
        .groupBy("attacker", "fromnpc").count().orderBy(F.desc("count")).show(25, truncate=False)
    
    print("Distribution of fromnpc:")
    tde_df.groupBy("fromnpc").count().orderBy(F.desc("count")).show(20, truncate=False)


AUDIT: mongo__tower_damage_events (Single-Pass)


26/11/13 05:41:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+----------+------------------+----------------+-------------+-----------+-------------+---------+-------------+--------------+----------------+---------------+--------------+-------------+----------+---------------------+-----------+------------+---------+----------------+------------------+---------------+
|total_rows|approx_distinct_id|distinct_matches|null_match_id|null_damage|nonpos_damage|null_time|negative_time|null_steam_acc|nonpos_steam_acc|steam_acc_64bit|null_byability|has_byability|has_byitem|both_ability_and_item|has_fromnpc|bad_attacker|bad_npcid|total_damage_sum|ability_damage_sum|item_damage_sum|
+----------+------------------+----------------+-------------+-----------+-------------+---------+-------------+--------------+----------------+---------------+--------------+-------------+----------+---------------------+-----------+------------+---------+----------------+------------------+---------------+
|241654288 |225552730         |315307          |0            |0       

+-----+--------+---------+
|npcid|events  |dmg      |
+-----+--------+---------+
|35   |14903015|555462162|
|25   |12999708|502227750|
|51   |10349216|479838600|
|30   |10220541|355189788|
|50   |9449449 |443952907|
|20   |9309082 |334714529|
|37   |8918714 |396607047|
|33   |8668220 |391983359|
|29   |8515596 |297886011|
|31   |8496466 |299071139|
|21   |8246590 |289511550|
|36   |8052462 |371215181|
|23   |7988857 |371762390|
|19   |7176778 |266460246|
|45   |6971224 |362236909|
|27   |6734009 |176440375|
|28   |6706885 |186313004|
|17   |6501470 |174299485|
|39   |6464365 |340510559|
|26   |6447284 |168040080|
|18   |6436058 |172513461|
|16   |6172692 |176651966|
|24   |6066037 |282120801|
|34   |6006055 |272529797|
|32   |5700453 |262074590|
+-----+--------+---------+
only showing top 25 rows
Top 25 attacker where steam_account_id is null or <= 0:
+--------+-------+-----+
|attacker|fromnpc|count|
+--------+-------+-----+
+--------+-------+-----+

Distribution of fromnpc:


+-------+---------+
|fromnpc|count    |
+-------+---------+
|0      |197044755|
|261    |10414426 |
|128    |5238758  |
|129    |4568119  |
|117    |4290286  |
|126    |4010571  |
|116    |2926473  |
|115    |2879989  |
|265    |2844119  |
|125    |1886848  |
|848    |905118   |
|190    |786627   |
|863    |677747   |
|189    |460701   |
|194    |393907   |
|150    |392936   |
|196    |286980   |
|195    |284628   |
|124    |178957   |
|106    |114143   |
+-------+---------+
only showing top 20 rows


### Расшифровка справочников `pg__game_modes` и `pg__lobby_types`

Выгружаем полные словари (26 режимов игры и 16 типов лобби) для подтверждения классификации: `mode_id = 23` — это `game_mode_turbo` (`balanced = false`), `mode_id = 22` - `game_mode_all_draft` (`balanced = true`).

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext("game_modes_and_lobbies") as spark:
    # Полные справочники режимов и лобби
    print("=== pg__game_modes ===")
    spark.table("lakehouse.raw.pg__game_modes").select("mode_id", "name", "balanced").orderBy("mode_id").show(50, truncate=False)
    
    print("=== pg__lobby_types ===")
    spark.table("lakehouse.raw.pg__lobby_types").select("lobby_id", "name", "balanced").orderBy("lobby_id").show(50, truncate=False)

=== pg__game_modes ===


+-------+--------------------------------+--------+
|mode_id|name                            |balanced|
+-------+--------------------------------+--------+
|0      |game_mode_unknown               |true    |
|1      |game_mode_all_pick              |true    |
|2      |game_mode_captains_mode         |true    |
|3      |game_mode_random_draft          |true    |
|4      |game_mode_single_draft          |true    |
|5      |game_mode_all_random            |true    |
|6      |game_mode_intro                 |false   |
|7      |game_mode_diretide              |false   |
|8      |game_mode_reverse_captains_mode |false   |
|9      |game_mode_greeviling            |false   |
|10     |game_mode_tutorial              |false   |
|11     |game_mode_mid_only              |false   |
|12     |game_mode_least_played          |true    |
|13     |game_mode_limited_heroes        |false   |
|14     |game_mode_compendium_matchmaking|false   |
|15     |game_mode_custom                |false   |
|16     |gam

### Проверка стабильности `npcid` в `mongo__building_events`

Проверяем, привязан ли `npcid` к конкретному типу строения, стороне, `maxhp` и координатам `(x, y)` глобально или зависит от матча, и сверяем со справочником `pg__npcs`.

**Инженерное решение:**
* В `mongo__building_events` (49,84 млн строк) каждому `npcid` (16–51) строго и неизменно соответствует одна стационарная постройка из `pg__npcs` (378 строк), причём здания появляются в логе до начала игры (`time = -89`).
* Дорогое соединение с `mongo__building_events` в ETL избыточно - для классификации строений достаточно компактного справочника `pg__npcs`.

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext("tower_id") as spark:
    
    # Проверка семантики npcid (pg__npcs vs mongo__building_events)
    npcs_df = spark.table("lakehouse.raw.pg__npcs").select("npc_id", "name", "team_name", "status_health")
    be_df = spark.table("lakehouse.raw.mongo__building_events").select(
        "match_id", "npcid", "type", "isradiant", "maxhp", "positionx", "positiony"
    )
    
    print("=== Стабильность (npcid -> тип/сторона/координаты) в building_events ===")
    be_summary = be_df.groupBy("npcid", "type", "isradiant", "maxhp").agg(
        F.count("*").alias("cnt"),
        F.min("positionx").alias("min_x"), F.max("positionx").alias("max_x"),
        F.min("positiony").alias("min_y"), F.max("positiony").alias("max_y")
    )
    be_summary.join(F.broadcast(npcs_df), be_summary.npcid == npcs_df.npc_id, "left") \
        .orderBy("npcid", F.desc("cnt")).show(100, truncate=False)

=== Стабильность (npcid -> тип/сторона/координаты) в building_events ===


+-----+--------+---------+-----+-------+-----+-----+-----+-----+------+-------------------------------+---------+-------------+
|npcid|type    |isradiant|maxhp|cnt    |min_x|max_x|min_y|max_y|npc_id|name                           |team_name|status_health|
+-----+--------+---------+-----+-------+-----+-----+-----+-----+------+-------------------------------+---------+-------------+
|16   |NULL    |NULL     |NULL |2248824|NULL |NULL |NULL |NULL |16    |npc_dota_goodguys_tower1_top   |GoodGuys |1800.0       |
|16   |TOWER   |true     |1800 |72211  |78   |78   |142  |142  |16    |npc_dota_goodguys_tower1_top   |GoodGuys |1800.0       |
|17   |NULL    |NULL     |NULL |2982117|NULL |NULL |NULL |NULL |17    |npc_dota_goodguys_tower1_mid   |GoodGuys |1800.0       |
|17   |TOWER   |true     |1800 |72211  |114  |114  |116  |116  |17    |npc_dota_goodguys_tower1_mid   |GoodGuys |1800.0       |
|18   |NULL    |NULL     |NULL |2748199|NULL |NULL |NULL |NULL |18    |npc_dota_goodguys_tower1_bot   |G

### Проверка доменов ключей и сопоставимости со справочниками

Проверяем уникальность положительных пар `(match_id, account_id)` и качество связей для `attacker`, `fromnpc`, `byability`, `byitem`, `pg__hero_abilities`.

**Выводы:**
* В `pg__match_players` найдено 237 598 дублирующихся пар `(match_id, account_id)` при `account_id > 0`** (причина размножения строк при прямом `JOIN`).
* 100% значений `attacker` присутствуют в `pg__heroes.hero_id` (0 несопоставленных).
* Топ `fromnpc` в `pg__npcs` - призванные существа (`wraith_king_skeleton_warrior`, `invoker_forged_spirit`, `broodmother_spiderling`, `shadow_shaman_ward_1..3`, `furion_treant`, `warlock_golem`, `lone_druid_bear1`), значительная часть которых наносит урон с `byability = 0`.
* 100% `byability > 0` есть в `pg__abilities` (0 потерь); в `pg__hero_abilities` каждая способность привязана максимум к 1 герою.
* `byitem = 223` (`Meteor Hammer`, 1 482 264 события) найден в `pg__items`, а `byitem = 371` (42 432 события) отсутствует в `pg__items` (требуется проверка `pg__neutral_items`).

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext("unique_match_acc_id") as spark:

    # Уникальность (match_id, account_id) в pg__match_players и пересечение ключей
    mp_df = spark.table("lakehouse.raw.pg__match_players").select("match_id", "account_id", "hero_id")
    matches_df = spark.table("lakehouse.raw.pg__matches").select("match_id", "duration", "game_mode")
    heroes_df = spark.table("lakehouse.raw.pg__heroes").select("hero_id", "localized_name")
    abilities_df = spark.table("lakehouse.raw.pg__abilities").select("ability_id", "name", "display_name")
    hero_ab_df = spark.table("lakehouse.raw.pg__hero_abilities").select("ability_id", "hero_id")
    items_df = spark.table("lakehouse.raw.pg__items").select("item_id", "localized_name")
    npcs_df = spark.table("lakehouse.raw.pg__npcs").select("npc_id", "name", "team_name", "status_health")
    
    print("=== Дубликаты (match_id, account_id) при account_id > 0 в pg__match_players ===")
    mp_dups = mp_df.filter(F.col("account_id") > 0).groupBy("match_id", "account_id").count().filter(F.col("count") > 1)
    print("Duplicate positive (match_id, account_id) pairs:", mp_dups.count())
    
    print("=== Проверка множества значений attacker, fromnpc, byability, byitem в фактах ===")
    tde_df = spark.table("lakehouse.raw.mongo__tower_damage_events").select(
        "id", "match_id", "time", "damage", "attacker", "byability", "byitem", "fromnpc", "npcid", "steam_account_id"
    )
    
    # Проверка attacker против pg__heroes и pg__npcs
    distinct_attackers = tde_df.groupBy("attacker").agg(F.count("*").alias("events"), F.sum("damage").alias("dmg"))
    print("Attacker NOT IN pg__heroes:")
    distinct_attackers.join(F.broadcast(heroes_df), distinct_attackers.attacker == heroes_df.hero_id, "left_anti").show(50, truncate=False)
    
    # Топ fromnpc с расшифровкой из pg__npcs
    print("Top 25 fromnpc joined with pg__npcs:")
    tde_df.filter(F.col("fromnpc") > 0).groupBy("fromnpc").agg(
        F.count("*").alias("events"), F.sum("damage").alias("dmg")
    ).join(F.broadcast(npcs_df), F.col("fromnpc") == F.col("npc_id"), "left") \
     .orderBy(F.desc("events")).show(25, truncate=False)
    
    # Топ byability с расшифровкой
    print("Top 25 byability joined with pg__abilities & count of non-matched abilities:")
    ab_stats = tde_df.filter(F.col("byability") > 0).groupBy("byability").agg(
        F.count("*").alias("events"), F.sum("damage").alias("dmg")
    )
    print("Unmatched byability IDs:", ab_stats.join(F.broadcast(abilities_df), ab_stats.byability == abilities_df.ability_id, "left_anti").count())
    ab_stats.join(F.broadcast(abilities_df), ab_stats.byability == abilities_df.ability_id, "left") \
        .orderBy(F.desc("dmg")).show(25, truncate=False)
    
    # Топ byitem с расшифровкой
    print("Top 15 byitem joined with pg__items:")
    tde_df.filter(F.col("byitem") > 0).groupBy("byitem").agg(
        F.count("*").alias("events"), F.sum("damage").alias("dmg")
    ).join(F.broadcast(items_df), F.col("byitem") == F.col("item_id"), "left") \
     .orderBy(F.desc("dmg")).show(15, truncate=False)
    
    # Проверка 1-к-М в pg__hero_abilities
    print("Abilities linked to >1 hero in pg__hero_abilities:")
    hero_ab_df.groupBy("ability_id").count().filter(F.col("count") > 1).show(20, truncate=False)

=== Дубликаты (match_id, account_id) при account_id > 0 в pg__match_players ===


Duplicate positive (match_id, account_id) pairs: 237598
=== Проверка множества значений attacker, fromnpc, byability, byitem в фактах ===
Attacker NOT IN pg__heroes:


[Stage 7:====================================================>    (12 + 1) / 13]

+--------+------+---+
|attacker|events|dmg|
+--------+------+---+
+--------+------+---+

Top 25 fromnpc joined with pg__npcs:


+-------+--------+---------+------+--------------------------------------------+---------+-------------+
|fromnpc|events  |dmg      |npc_id|name                                        |team_name|status_health|
+-------+--------+---------+------+--------------------------------------------+---------+-------------+
|261    |10484861|107434999|261   |npc_dota_wraith_king_skeleton_warrior       |GoodGuys |175.0        |
|128    |5265882 |150859232|128   |npc_dota_invoker_forged_spirit              |GoodGuys |5.0          |
|129    |4587679 |26678077 |129   |npc_dota_broodmother_spiderling             |GoodGuys |325.0        |
|117    |4319772 |91452552 |117   |npc_dota_shadow_shaman_ward_3               |BadGuys  |0.0          |
|126    |4037460 |73492879 |126   |npc_dota_furion_treant                      |GoodGuys |550.0        |
|116    |2945324 |44337215 |116   |npc_dota_shadow_shaman_ward_2               |BadGuys  |0.0          |
|115    |2901713 |25213987 |115   |npc_dota_shadow_sham

Unmatched byability IDs: 0


+---------+--------+---------+----------+------------------------------------+-----------------+
|byability|events  |dmg      |ability_id|name                                |display_name     |
+---------+--------+---------+----------+------------------------------------+-----------------+
|5223     |7857067 |157576079|5223      |luna_moon_glaive                    |Moon Glaives     |
|5087     |10484850|107434903|5087      |skeleton_king_bone_guard            |Bone Guard       |
|5229     |2439287 |65451786 |5229      |dragon_knight_elder_dragon_form     |Elder Dragon Form|
|5186     |261757  |57508658 |5186      |pugna_nether_blast                  |Nether Blast     |
|5024     |246598  |50188113 |5024      |earthshaker_enchant_totem           |Enchant Totem    |
|5093     |4011347 |45984589 |5093      |death_prophet_exorcism              |Exorcism         |
|5242     |1245150 |39320234 |5242      |leshrac_diabolic_edict              |Diabolic Edict   |
|5260     |2409742 |38010792 |

+------+-------+--------+-------+--------------+
|byitem|events |dmg     |item_id|localized_name|
+------+-------+--------+-------+--------------+
|223   |1482264|92332883|223    |Meteor Hammer |
|371   |42432  |3075391 |NULL   |NULL          |
+------+-------+--------+-------+--------------+

Abilities linked to >1 hero in pg__hero_abilities:
+----------+-----+
|ability_id|count|
+----------+-----+
+----------+-----+



### Измерение потерь при наивном `JOIN` и точная проверка дубликатов `id`

Оцениваем покрытие при прямом соединении с `pg__matches` и `pg__match_players` по `(match_id, steam_account_id)` и выполняем точную группировку по `id`.

**Выводы:**
* Дубликатов по `id` нет (`distinct_duplicated_ids = 0`)**: расхождение HyperLogLog на шаге 3 было погрешностью аппроксимации.
* Из-за дублей `account_id > 0` прямой `JOIN` с `pg__match_players` размножил таблицу до 243 246 229 строк (+1,59 млн лишних записей).
* При этом 47 044 439 событий (~19,3%, на 1,96 млрд урона) не сматчились по `(match_id, steam_account_id)`, так как реплей хранит реальный ID игрока, а Steam API в `pg__match_players` отдаёт заглушку скрытого профиля.
* Внутри сматченных строк выявлено 3 426 случаев расхождения `attacker != mp_hero_id`** — это обосновывает приоритет `account_id` перед `hero_id` в будущем каскадном связывании.
* Событий с `time > duration` нет.

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext("join_coverage") as spark:

    # Join Coverage: match_id -> pg__matches и (match_id, steam_account_id) -> pg__match_players
    print("=== Join Coverage по match_id и (match_id, steam_account_id) ===")
    matches_df = spark.table("lakehouse.raw.pg__matches").select("match_id", "duration", "game_mode")
    mp_df = spark.table("lakehouse.raw.pg__match_players").select("match_id", "account_id", "hero_id")
    tde_df = spark.table("lakehouse.raw.mongo__tower_damage_events").select(
        "id", "match_id", "time", "damage", "attacker", "byability", "byitem", "fromnpc", "npcid", "steam_account_id"
    )
    tde_with_match = tde_df.join(
        F.broadcast(matches_df.select("match_id", F.col("duration").alias("m_duration"))),
        on="match_id",
        how="left"
    ).join(
        mp_df.select("match_id", F.col("account_id").alias("steam_account_id"), F.col("hero_id").alias("mp_hero_id")),
        on=["match_id", "steam_account_id"],
        how="left"
    )
    
    tde_with_match.agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("m_duration").isNull(), 1).otherwise(0)).alias("unmatched_match_events"),
        F.sum(F.when(F.col("m_duration").isNull(), F.col("damage")).otherwise(0).cast("long")).alias("unmatched_match_dmg"),
        F.sum(F.when(F.col("mp_hero_id").isNull(), 1).otherwise(0)).alias("unmatched_player_events"),
        F.sum(F.when(F.col("mp_hero_id").isNull(), F.col("damage")).otherwise(0).cast("long")).alias("unmatched_player_dmg"),
        F.sum(F.when((F.col("mp_hero_id").isNotNull()) & (F.col("attacker") != F.col("mp_hero_id")), 1).otherwise(0)).alias("attacker_neq_mp_hero"),
        F.sum(F.when(F.col("m_duration").isNotNull() & (F.col("time") > F.col("m_duration")), 1).otherwise(0)).alias("time_after_duration")
    ).show(truncate=False)
    
    # Точная проверка дубликатов по id в mongo__tower_damage_events
    print("=== Точная проверка дубликатов по id ===")
    dup_ids = tde_df.groupBy("id").count().filter(F.col("count") > 1)
    dup_summary = dup_ids.agg(
        F.count("*").alias("distinct_duplicated_ids"),
        F.sum(F.col("count") - 1).alias("extra_duplicate_rows")
    )
    dup_summary.show(truncate=False)

=== Join Coverage по match_id и (match_id, steam_account_id) ===


+------------+----------------------+-------------------+-----------------------+--------------------+--------------------+-------------------+
|total_events|unmatched_match_events|unmatched_match_dmg|unmatched_player_events|unmatched_player_dmg|attacker_neq_mp_hero|time_after_duration|
+------------+----------------------+-------------------+-----------------------+--------------------+--------------------+-------------------+
|243246229   |1181751               |50806557           |47044439               |1955623285          |3426                |0                  |
+------------+----------------------+-------------------+-----------------------+--------------------+--------------------+-------------------+

=== Точная проверка дубликатов по id ===


+-----------------------+--------------------+
|distinct_duplicated_ids|extra_duplicate_rows|
+-----------------------+--------------------+
|0                      |NULL                |
+-----------------------+--------------------+



### Проектирование каскадного ключа, полная классификация `npcid` и сверка `tower_damage`

Закрываем последние вопросы перед построением `STAGE`: природу дублей в `pg__match_players`, полный список `npcid`, матрицу источников урона, эффективность каскадного `JOIN` и семантику `pg__match_players.tower_damage`.

**Ключевые результаты:**
* Дубли `(match_id, account_id)` в `pg__match_players` вызваны ровно двумя заглушками скрытых профилей: `4294967295` (237 598 матчей, 1 499 162 строки) и `0` (8 953 матча, 76 960 строк). Пары `(match_id, hero_id)` уникальны в 99,97% матчей (лишь 134 дубля).
* Полная расшифровка `npcid` (16–51): таблица событий включает башни (`16–35`), филлеры базы (`36–37`), казармы (`38–49`) и трон (`50–51`).
* Матрица источников (на 243 461 602 события):** автоатаки - 174 063 866 событий (8,64 млрд урона), призванные юниты без `byability` - 32 008 202 события (624,14 млн урона), прямые способности - 22 950 005, способность + саммон - 12 913 684, предметы - 1 525 845.
* Эффективность каскадного `JOIN`: по уникальному `account_id` связано 196 218 434 события (80,60%), спасено по fallback `(match_id, attacker = hero_id)` - 46 578 348 событий (19,13%), несопоставленных осталось 664 820 (0,27%).
* Сверка с `pg__match_players.tower_damage` (на 5 000 матчах): сумма событий `TOWER (85 111 977) + BARRACKS (38 928 963) + FORT (13 615 076) = 137 656 016`, что совпадает с `pg_tower_damage_sum = 137 730 920` с точностью до 0,05%. Вспомогательные здания `FILLER` (12,13 млн) в официальный агрегат `tower_damage` не входят.

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext("Final") as spark:
    
    # Инспекция pg__neutral_items и причин дублей в pg__match_players
    print("=== pg__neutral_items schema & sample ===")
    ni_df = spark.table("lakehouse.raw.pg__neutral_items")
    ni_df.printSchema()
    ni_df.show(10, truncate=False)
    
    mp_df = spark.table("lakehouse.raw.pg__match_players").select(
        "match_id", "account_id", "player_slot", "hero_id", "is_radiant", "win", "tower_damage"
    )
    
    print("=== Топ дублирующихся account_id в (match_id, account_id) ===")
    mp_df.groupBy("match_id", "account_id").count().filter(F.col("count") > 1) \
        .groupBy("account_id").agg(
            F.count("*").alias("dup_matches"),
            F.sum("count").alias("total_dup_rows")
        ).orderBy(F.desc("dup_matches")).show(10, truncate=False)
    
    print("=== Уникальность (match_id, hero_id) в pg__match_players ===")
    mp_df.filter(F.col("hero_id") > 0).groupBy("match_id", "hero_id").count().filter(F.col("count") > 1) \
        .agg(F.count("*").alias("dup_match_hero_pairs")).show()
    
    # Полный список всех npcid в mongo__tower_damage_events + расшифровка pg__npcs
    tde_df = spark.table("lakehouse.raw.mongo__tower_damage_events").select(
        "match_id", "damage", "attacker", "byability", "byitem", "fromnpc", "npcid", "steam_account_id"
    )
    npcs_df = spark.table("lakehouse.raw.pg__npcs").select("npc_id", "name", "team_name", "status_health")
    
    print("=== Все уникальные npcid в фактах и их тип постройки ===")
    tde_df.groupBy("npcid").agg(
        F.count("*").alias("events"),
        F.sum(F.col("damage").cast("long")).alias("total_dmg")
    ).join(F.broadcast(npcs_df), F.col("npcid") == F.col("npc_id"), "left") \
     .orderBy("npcid").show(100, truncate=False)
    
    # Матрица источников урона: (byability > 0) x (byitem > 0) x (fromnpc > 0)
    print("=== Матрица (byability>0, byitem>0, fromnpc>0) ===")
    tde_df.groupBy(
        (F.col("byability") > 0).alias("has_ability"),
        (F.col("byitem") > 0).alias("has_item"),
        (F.col("fromnpc") > 0).alias("has_fromnpc")
    ).agg(
        F.count("*").alias("events"),
        F.sum(F.col("damage").cast("long")).alias("total_dmg")
    ).orderBy(F.desc("events")).show(truncate=False)
    
    # Проверка каскадного JOIN с pg__match_players (без размножения строк!)
    # Берем только уникальные пары (match_id, account_id) и (match_id, hero_id)
    mp_by_acc = mp_df.groupBy("match_id", "account_id").agg(
        F.count("*").alias("cnt"),
        F.first("hero_id").alias("mp_hero_id")
    ).filter(F.col("cnt") == 1).drop("cnt")
    
    mp_by_hero = mp_df.filter(F.col("hero_id") > 0).groupBy("match_id", "hero_id").agg(
        F.count("*").alias("cnt"),
        F.first("account_id").alias("mp_acc_by_hero")
    ).filter(F.col("cnt") == 1).drop("cnt")
    
    print("=== Покрытие при каскадном JOIN: 1) (match_id, steam_account_id) -> 2) (match_id, attacker=hero_id) ===")
    tde_joined = tde_df.join(
        mp_by_acc.withColumnRenamed("account_id", "steam_account_id"),
        on=["match_id", "steam_account_id"],
        how="left"
    ).join(
        mp_by_hero.withColumnRenamed("hero_id", "attacker"),
        on=["match_id", "attacker"],
        how="left"
    )
    
    tde_joined.agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("mp_hero_id").isNotNull(), 1).otherwise(0)).alias("matched_by_account"),
        F.sum(F.when(F.col("mp_hero_id").isNull() & F.col("mp_acc_by_hero").isNotNull(), 1).otherwise(0)).alias("rescued_by_hero"),
        F.sum(F.when(F.col("mp_hero_id").isNull() & F.col("mp_acc_by_hero").isNull(), 1).otherwise(0)).alias("still_unmatched_events"),
        F.sum(F.when(F.col("mp_hero_id").isNull() & F.col("mp_acc_by_hero").isNull(), F.col("damage")).otherwise(0).cast("long")).alias("still_unmatched_dmg")
    ).show(truncate=False)
    
    # Семантика pg__match_players.tower_damage (башни vs башни+бараки+трон)
    print("=== Сравнение SUM(tower_damage) в pg__match_players и SUM(damage) из событий по сматченным матчам ===")
    # Посчитаем по 1000 случайным матчам из pg__matches (быстро и наглядно)
    sample_matches = spark.table("lakehouse.raw.pg__matches").select("match_id").limit(5000)
    
    ev_match_dmg = tde_df.join(F.broadcast(sample_matches), "match_id").agg(
        F.sum(F.when(F.col("npcid").between(16, 35), F.col("damage")).otherwise(0).cast("long")).alias("ev_towers_only_16_35"),
        F.sum(F.when(F.col("npcid").between(38, 49), F.col("damage")).otherwise(0).cast("long")).alias("ev_barracks_38_49"),
        F.sum(F.when(F.col("npcid").isin(50, 51), F.col("damage")).otherwise(0).cast("long")).alias("ev_fort_50_51"),
        F.sum(F.col("damage").cast("long")).alias("ev_all_buildings")
    )
    
    mp_match_dmg = mp_df.join(F.broadcast(sample_matches), "match_id").agg(
        F.sum(F.col("tower_damage").cast("long")).alias("pg_tower_damage_sum")
    )
    
    ev_match_dmg.crossJoin(mp_match_dmg).show(truncate=False)

=== pg__neutral_items schema & sample ===
root
 |-- description: string (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- localized_name: string (nullable = true)
 |-- lore: string (nullable = true)
 |-- name: string (nullable = true)
 |-- tier: integer (nullable = true)
 |-- job_id: string (nullable = true)
 |-- sys_insert_stamp: timestamp (nullable = true)
 |-- src_modified_stamp: timestamp (nullable = true)



+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+--------------------+--------------------------------------------------------------------------------------+--------------------+----+------------------------------------+--------------------------+------------------+
|description                                                                                                                                                                                                                                                                                                    |item_id|localized_name      |lore                                                                                  |name                |tier|job_id                

+----------+-----------+--------------+
|account_id|dup_matches|total_dup_rows|
+----------+-----------+--------------+
|4294967295|237598     |1499162       |
|0         |8953       |76960         |
+----------+-----------+--------------+

=== Уникальность (match_id, hero_id) в pg__match_players ===


+--------------------+
|dup_match_hero_pairs|
+--------------------+
|                 134|
+--------------------+

=== Все уникальные npcid в фактах и их тип постройки ===


+-----+--------+---------+------+-------------------------------+---------+-------------+
|npcid|events  |total_dmg|npc_id|name                           |team_name|status_health|
+-----+--------+---------+------+-------------------------------+---------+-------------+
|16   |6216409 |177928966|16    |npc_dota_goodguys_tower1_top   |GoodGuys |1800.0       |
|17   |6549146 |175568283|17    |npc_dota_goodguys_tower1_mid   |GoodGuys |1800.0       |
|18   |6484309 |173769434|18    |npc_dota_goodguys_tower1_bot   |GoodGuys |1800.0       |
|19   |7229916 |268444656|19    |npc_dota_goodguys_tower2_top   |GoodGuys |2500.0       |
|20   |9375686 |337115446|20    |npc_dota_goodguys_tower2_mid   |GoodGuys |2500.0       |
|21   |8309290 |291709623|21    |npc_dota_goodguys_tower2_bot   |GoodGuys |2500.0       |
|22   |5093894 |242371079|22    |npc_dota_goodguys_tower3_top   |GoodGuys |2500.0       |
|23   |8046215 |374461282|23    |npc_dota_goodguys_tower3_mid   |GoodGuys |2500.0       |
|24   |611

+-----------+--------+-----------+---------+----------+
|has_ability|has_item|has_fromnpc|events   |total_dmg |
+-----------+--------+-----------+---------+----------+
|false      |false   |false      |174063866|8637001888|
|false      |false   |true       |32008202 |624135154 |
|true       |false   |false      |22950005 |630082703 |
|true       |false   |true       |12913684 |149213765 |
|false      |true    |false      |1525367  |95449249  |
|false      |true    |true       |478      |30362     |
+-----------+--------+-----------+---------+----------+

=== Покрытие при каскадном JOIN: 1) (match_id, steam_account_id) -> 2) (match_id, attacker=hero_id) ===


+------------+------------------+---------------+----------------------+-------------------+
|total_events|matched_by_account|rescued_by_hero|still_unmatched_events|still_unmatched_dmg|
+------------+------------------+---------------+----------------------+-------------------+
|243461602   |196218434         |46578348       |664820                |28746024           |
+------------+------------------+---------------+----------------------+-------------------+

=== Сравнение SUM(tower_damage) в pg__match_players и SUM(damage) из событий по сматченным матчам ===


+--------------------+-----------------+-------------+----------------+-------------------+
|ev_towers_only_16_35|ev_barracks_38_49|ev_fort_50_51|ev_all_buildings|pg_tower_damage_sum|
+--------------------+-----------------+-------------+----------------+-------------------+
|85111977            |38928963         |13615076     |149789603       |137730920          |
+--------------------+-----------------+-------------+----------------+-------------------+



### Матрица пересечения `match_id` между источниками

Фиксируем итоговое покрытие матчей между `pg__matches`, `pg__match_players` и `mongo__tower_damage_events`:
* **316 254 матча** - полноценное ядро исследования (есть и в `pg__matches`, и в `pg__match_players`, и в Mongo).
* **150 929 матчей** - сиротские записи в `pg__match_players` без карточки матча в `pg__matches` и без событий в Mongo (сохраняются в `STAGE` с флагом `in_pg_matches = false` для отчёта качества и отсекаются в `MART`).
* **856 матчей** - есть в Postgres, но без событий урона по строениям в Mongo.
* **909 матчей** - есть события в Mongo и игроки, но отсутствует запись в `pg__matches`.

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

with SparkSessionContext("Intersection_check") as spark:

    # Проверка пересечения match_id между таблицами pg__match_players, pg__matches и mongo__tower_damage_events
    matches_ids = spark.table("lakehouse.raw.pg__matches").select("match_id").distinct()
    mp_ids = spark.table("lakehouse.raw.pg__match_players").select("match_id").distinct()
    tde_ids = spark.table("lakehouse.raw.mongo__tower_damage_events").select("match_id").distinct()
    
    print("Пересечение match_id между таблицами:")
    mp_ids.join(F.broadcast(matches_ids.withColumn("in_matches", F.lit(1))), "match_id", "left") \
        .join(tde_ids.withColumn("in_tde", F.lit(1)), "match_id", "left") \
        .groupBy(F.col("in_matches").isNotNull().alias("in_pg_matches"),
                 F.col("in_tde").isNotNull().alias("in_mongo_tde")) \
        .count().show()

Пересечение match_id между таблицами:


+-------------+------------+------+
|in_pg_matches|in_mongo_tde| count|
+-------------+------------+------+
|         true|       false|   856|
|        false|        true|   909|
|        false|       false|150929|
|         true|        true|316254|
+-------------+------------+------+

